In [ ]:
# Reader scale: frozen Kaggle stack and private HF credential.
import os, shutil, sys
from pathlib import Path
secret_value_0 = os.environ.get('HF_TOKEN')
if not secret_value_0:
    try:
        from kaggle_secrets import UserSecretsClient
        secret_value_0 = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
pass
print('python', sys.version.split()[0], 'working disk', shutil.disk_usage('/kaggle/working'))
print('input mounts', sorted(str(p) for p in Path('/kaggle/input').iterdir()))


In [ ]:
# T4 dependencies
import subprocess, sys
try:
    _cap=subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],capture_output=True,text=True,timeout=60).stdout.strip().splitlines()[0].strip()
except Exception: _cap=''
print('compute_cap:', _cap or 'unknown')
%pip install -q "transformers==5.17.0" datasets safetensors huggingface_hub matplotlib accelerate
import torch
assert torch.cuda.is_available(), 'need a GPU accelerator'
assert torch.cuda.get_device_capability(0) == (7, 5)
print('torch', torch.__version__, '| cap', torch.cuda.get_device_capability(0))
import transformers, datasets, safetensors, huggingface_hub
print(transformers.__version__, datasets.__version__, safetensors.__version__, huggingface_hub.__version__)

In [ ]:
# Cell 2 — single config block (edit here only)
from dataclasses import dataclass

@dataclass(frozen=True)
class Cfg:
    SOURCE_ID: str = 'Qwen/Qwen3.8-Flash-Next-FP8'
    TARGET_ID: str = 'Qwen/Qwen3.5-2B'
    MEM_DIM: int = 2560
    HIDDEN: int = 2048
    N_LAYERS: int = 24
    NGRAM: int = 3
    HEADS_PER_NGRAM: int = 8   # 16 address slots per token
    ROW_DIM: int = 160         # per-slot row dim; 16*160=2560
    ROWS_PER_PART: int = 2_500_012
    VOCAB_BASE: int = 20_000_000
    SEED: int = 1234           # PLE hash seed — never change
    EOS: int = 248044  # config <|endoftext|>: PLE-training terminator (NOT chat <|im_end|> 248046)
    VOCAB: int = 248320
    SEQ: int = 512
    VAL_FAST_TOKENS: int = 65536    # 128 x 512
    VAL_FULL_TOKENS: int = 524288   # 1024 x 512 (35B parity)
    DATASET_ID: str = 'HuggingFaceFW/fineweb-edu'  # FIRST experiment: FineWeb-Edu ONLY
    DATASET_CONFIG: str = 'sample-10BT'
    SMOKE_TOKENS: int = 5120    # correctness pass only (10 x 512); sweep stays OFF
    LR: float = 3e-5
    WD: float = 0.01
    WARMUP_FRAC: float = 0.05
    CKPTS: tuple = (100000, 250000, 500000)  # threshold-crossing (not exact multiples)
    PLACEMENTS: tuple = ((2, 8),)  # zero-based IDX, 35B convention
    BRANCHES: int = 1
    GAMMA_INIT: float = 1e-3   # 0.0 = identity test mode
    SWEEP_ENABLED: bool = False  # DONE: 500K sweep complete, winner IDX 2+8
    BRANCH_ENABLED: bool = False  # DONE: controls persisted to ninnix/qwen-ple-reader-checkpoints
    REAL1M_ENABLED: bool = False  # DONE: v23 COMPLETE, bundles in dataset v4
    REAL5M_ENABLED: bool = False  # STAGED: 1M->2M->3M->5M legs, early-stop on saturate/reverse
    STAGEA_ENABLED: bool = False  # eval lives in kaggle_qwen35_08b_ple_eval.ipynb, never here

C = Cfg()
def describe(layers): return f"IDX {list(layers)} = HUMAN {[l+1 for l in layers]}"
print(C)
print('placements:', ' / '.join(describe(l) for l in C.PLACEMENTS))
for sites, R in [(2,1)]:
    n = sites*(R*C.MEM_DIM*C.HIDDEN + C.MEM_DIM*C.HIDDEN) + sites*R + sites
    print(f'sites={sites} R={R}: ~{n/1e6:.2f}M trainable')


In [ ]:
# Cell 3 — EXACT source addressing (port of src/qwen36_ple/hashing.py — do not modify)
import math
import torch

MASK64=(1<<64)-1; GAMMA=0x9E3779B97F4A7C15; M1=0xBF58476D1CE4E5B9; M2=0x94D049BB133111EB; LPRIME=10007

def splitmix64(v):
    v=(v+GAMMA)&MASK64; v=((v^(v>>30))*M1)&MASK64; v=((v^(v>>27))*M2)&MASK64; return (v^(v>>31))&MASK64

def layer_multipliers(vocab, ngram, ple_idx=0, seed=1234):
    mmax=((1<<63)-1)//max(vocab,1); hb=max(1,mmax//2); base=seed+LPRIME*ple_idx
    return tuple(2*(splitmix64((base+GAMMA*(i+1))&MASK64)%hb)+1 for i in range(ngram))

def is_prime(v):
    if v<2: return False
    if v%2==0: return v==2
    return all(v%d for d in range(3, math.isqrt(v)+1, 2))

def nth_prime_after(s, k):
    p=s
    for _ in range(k):
        p+=1
        while not is_prime(p): p+=1
    return p

def head_layout(ngram=3, hpn=8, base=20_000_000, ple_idx=0):
    n=(ngram-1)*hpn
    sizes=tuple(nth_prime_after(base-1, ple_idx*n+h+1) for h in range(n))
    off=[]; t=0
    for s in sizes: off.append(t); t+=s
    return sizes, tuple(off)

def shift_right_ignore_eos(ids, shift, eos):
    if shift==0: return ids
    B,L=ids.shape; pos=torch.arange(L, device=ids.device)
    eos_pos=torch.where(ids==eos, pos, -1); prev_inc=torch.cummax(eos_pos,1).values
    prev=torch.cat([eos_pos.new_full((B,1),-1), prev_inc[:,:-1]],1)
    inseg=pos.unsqueeze(0)-(prev+1); src=pos-shift
    sh=ids.gather(1, src.clamp_min(0).unsqueeze(0).expand(B,-1))
    valid=(inseg>=shift)&(src.unsqueeze(0)>=0)
    return torch.where(valid, sh, ids.new_full((), eos))

def ngram_indices(ids, eos_token_id=248044, vocab_size=248320, ngram_size=3, heads_per_ngram=8,
                    vocab_size_base=20_000_000, ple_layer_index=0, seed=1234):
    ids=ids.long()
    mult=torch.tensor(layer_multipliers(vocab_size, ngram_size, ple_layer_index, seed), device=ids.device)
    sizes, offs=head_layout(ngram_size, heads_per_ngram, vocab_size_base, ple_layer_index)
    sizes=torch.tensor(sizes, device=ids.device); offs=torch.tensor(offs, device=ids.device)
    sh=[shift_right_ignore_eos(ids,s,eos_token_id) for s in range(ngram_size)]
    blocks=[]
    for ng in range(2, ngram_size+1):
        st=(ng-2)*heads_per_ngram; mixed=sh[0]*mult[0]
        for p in range(1,ng): mixed=torch.bitwise_xor(mixed, sh[p]*mult[p])
        blocks.append(torch.remainder(mixed.unsqueeze(-1), sizes[st:st+heads_per_ngram])+offs[st:st+heads_per_ngram])
    return torch.cat(blocks,-1)  # [B,L,16] GLOBAL PLE addresses, never raw token ids

_a=ngram_indices(torch.tensor([[1,2,3,4,5]])); _b=ngram_indices(torch.tensor([[1,2,3,4,5]]))
assert torch.equal(_a,_b) and _a.shape==(1,5,16)
SIZES, OFFS = head_layout()
print('hash ok; slots/head addrs e.g.', tuple(_a[0,2,:4].tolist()), '| head0 range', (OFFS[0], OFFS[0]+SIZES[0]))
def addresses(token_cpu):
    '''ONLY path from tokens to PLE rows: exact source hashes -> global head addresses.'''
    return ngram_indices(token_cpu, eos_token_id=C.EOS, vocab_size=C.VOCAB,
                         ngram_size=C.NGRAM, heads_per_ngram=C.HEADS_PER_NGRAM,
                         vocab_size_base=C.VOCAB_BASE, ple_layer_index=0, seed=C.SEED)

In [ ]:
# Cell 4 — shared-value reader (hidden=2048; reductions stay FP32 so fp16 backbone is safe)
import math
import torch
from torch import nn

def rms_norm(x, eps=1e-6):  # always FP32 reduction, cast back: fp16-safe
    return x.float().mul(torch.rsqrt(x.float().square().mean(-1, keepdim=True)+eps)).to(x.dtype)

class SharedValueReader(nn.Module):
    def __init__(self, mem_dim=2560, hidden=2048, branches=1, gamma=0.0):
        super().__init__(); self.mem_dim=mem_dim; self.hidden=hidden; self.branches=branches
        self.keys=nn.ModuleList(nn.Linear(mem_dim, hidden, bias=False) for _ in range(branches))
        self.value=nn.Linear(mem_dim, hidden, bias=False)
        self.beta=nn.Parameter(torch.zeros(branches))
        self.gamma=nn.Parameter(torch.tensor(float(gamma)))
        self.last_gate=None
    def stats(self):
        if self.last_gate is None: return None
        g=self.last_gate.float()
        return {'mean':g.mean().item(),'std':g.std(correction=0).item(),'near_zero':(g<0.01).float().mean().item()}
    def forward(self, h, m):
        assert h.shape[:-1]==m.shape[:-1], (h.shape, m.shape)
        h_dtype=h.dtype; h=h.float(); m=m.float()  # backbone may be fp16; reader computes FP32
        q=rms_norm(h); v=self.value(m); gs=[]
        for b,proj in enumerate(self.keys):
            k=rms_norm(proj(m))
            s=(q.float()*k.float()).sum(-1)/math.sqrt(self.hidden)
            gs.append(torch.sigmoid(s+self.beta[b].float()).to(v.dtype))
        g=torch.stack(gs,0)
        o=(g.unsqueeze(-1)*v.unsqueeze(0)).mean(0)
        self.last_gate=g.detach()
        return (h+self.gamma*o).to(h_dtype)  # residual back to backbone dtype; params stay FP32

_r=SharedValueReader(gamma=0.0); _h=torch.randn(1,4,2048); _m=torch.randn(1,4,2560)
assert torch.equal(_r(_h,_m),_h)
print('reader identity ok; R=1 params:', sum(p.numel() for p in _r.parameters()))

In [ ]:
# Cell 5 — injection hooks (IDX convention) + layer helper
import torch
from torch import nn

def decoder_layers(model):
    for path in ['model.layers','language_model.layers','transformer.h']:
        o=model
        try:
            for a in path.split('.'): o=getattr(o,a)
            if len(o)==24 or len(o)>0: return o
        except Exception: pass
    raise RuntimeError('decoder layers not found')

class ReaderInjection(nn.Module):
    '''layers = zero-based IDX list, exactly like the 35B run (e.g. (2,) = third block).'''
    def __init__(self, model, layers, mem_dim=2560, hidden=2048, branches=1, gamma=0.0):
        super().__init__()
        self.idx=tuple(layers)
        self.readers=nn.ModuleDict({str(l):SharedValueReader(mem_dim,hidden,branches,gamma) for l in layers})
        self.memory=None; self.handles=[]
        dec=decoder_layers(model)
        assert len(dec)==C.N_LAYERS, f'decoder count {len(dec)} != {C.N_LAYERS} — wrong hook target'
        for l in layers:
            self.handles.append(dec[l].register_forward_pre_hook(self._hook(str(l)), with_kwargs=True))
        print(f'inject at IDX {list(layers)} = HUMAN {[l+1 for l in layers]}')
    def _hook(self,name):
        def fn(mod,args,kw):
            if self.memory is None: return args,kw
            m=self.memory
            L=args[0].shape[1] if args else kw['hidden_states'].shape[1]
            if m.shape[1]!=L: m=m[:,:L]
            if args: return (self.readers[name](args[0],m),*args[1:]),kw
            kw['hidden_states']=self.readers[name](kw['hidden_states'],m); return args,kw
        return fn
    def set_memory(self,m): self.memory=m
    def close(self):
        [h.remove() for h in self.handles]; self.handles.clear()

print('injection ok')

In [ ]:
# Cell 6 — PLE stores: real (mount-only, exact scale or abort) + calibrated controls
import json, os
from pathlib import Path
import torch
from safetensors import safe_open

PLE_TMPL='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.shard_{p}.weight'
PLE_SCALE='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.weight_scale'

def rss_mb():
    '''Host RSS in MiB (Linux /proc; -1 if unavailable). Proves bounded RAM.'''
    try:
        with open('/proc/self/status') as _f:
            for _line in _f:
                if _line.startswith('VmRSS:'): return float(_line.split()[1])/1024
    except Exception: pass
    try:
        import resource; return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024
    except Exception: return -1.0

def find_ple_manifests():
    hits=sorted(Path('/kaggle/input').rglob('manifest.json'))
    out=[]
    for h in hits:
        try:
            m=json.loads(h.read_text())
            if isinstance(m, dict) and 'parts' in m: out.append(h)
        except Exception: pass
    return out

class MountPLE:
    '''Real frozen PLE (row-level mmap, 35B-validated). REQUIRES /kaggle/input mount. Never downloads. Never caches parts: each lookup fetches ONLY needed rows via safetensors get_slice runs, so host RAM stays bounded after all 128 parts are touched.'''
    def __init__(self, manifest=None):
        manifests=[Path(manifest)] if manifest else find_ple_manifests()
        if not manifests or not all(p.exists() for p in manifests):
            raise RuntimeError('Real-PLE ABORT: no /kaggle/input PLE dataset attached. Attach the pinned shards+manifest.json dataset(s) first; refusing to download 48.7 GiB into /kaggle/working.')
        self.part_paths={}; revs=set()
        for mp in manifests:
            m=json.loads(mp.read_text())
            if not revs: self.rpp=m.get('rows_per_part',2500012); self.rd=m.get('row_dim',160)
            revs.add(m.get('ple_revision','unknown'))
            for k,v in m['parts'].items():
                p=mp.parent/v
                if not p.exists(): continue  # each dataset holds only its own shards
                if int(k) in self.part_paths:
                    assert self.part_paths[int(k)].name==p.name, f'part {k} filename clash'
                    continue
                self.part_paths[int(k)]=p
        assert len(revs)==1, f'mixed PLE revisions: {revs}'
        self.ple_revision=revs.pop()
        assert len(self.part_paths)==128, f'need all 128 parts, have {len(self.part_paths)}'
        missing=[str(p) for p in self.part_paths.values() if not p.exists()]
        if missing: raise RuntimeError(f"Real-PLE ABORT: {len(missing)} shard files missing, e.g. {missing[0]}")
        self.scale=self._resolve_scale()  # exact scale or abort — no fallback constant
        self.calls=0; self.rows_read=0
        self.parts_touched=set()  # cumulative DISTINCT parts; no part tensors ever held
        print(f'mounted PLE rev={self.ple_revision} parts={len(self.part_paths)} scale={self.scale}')
    def _resolve_scale(self):
        for f in sorted(set(self.part_paths.values())):
            try:
                with safe_open(f, framework='pt', device='cpu') as fh:
                    if PLE_SCALE in fh.keys():
                        return float(fh.get_tensor(PLE_SCALE).float().mean())
            except Exception: pass
        raise RuntimeError('Real-PLE ABORT: weight_scale tensor not found in pinned shards/index. Refusing hardcoded fallback.')
    def stats(self):
        return {'calls': self.calls, 'rows_read': self.rows_read,
                'parts_touched': len(self.parts_touched), 'held_part_tensors': 0,
                'scale': self.scale, 'rss_MiB': round(rss_mb(), 1)}
    @staticmethod
    def tensor_name(part): return PLE_TMPL.format(p=part)
    def lookup(self, indices):  # indices = GLOBAL head addresses [..,16] from ngram_indices()
        '''Row-level mmap reads: group deduped addresses by part (sorted), fetch ONLY
        needed rows as contiguous get_slice runs, dequantize the gathered rows. Full
        part tensors (~381 MiB each) are never materialized or cached.'''
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)  # dedup repeated addresses
        parts=torch.div(uniq, self.rpp, rounding_mode='floor'); local=uniq%self.rpp
        table=torch.empty(uniq.numel(), self.rd, dtype=torch.float32)
        for p in torch.unique(parts).tolist():  # ascending part order (page-friendly)
            pos=torch.nonzero(parts==p).flatten()
            lrows=local.index_select(0, pos); srows, sidx=torch.sort(lrows)
            runs=[]; a=int(srows[0]); prev=a
            for r in srows[1:].tolist():
                if r==prev+1: prev=r
                else: runs.append((a, prev+1)); a=prev=r
            runs.append((a, prev+1))
            path=self.part_paths.get(p)
            if path is None: raise FileNotFoundError(f'PLE part {p} not in manifest')
            with safe_open(str(path), framework='pt', device='cpu') as fh:
                sl=fh.get_slice(self.tensor_name(p))
                got=torch.cat([sl[x:y].to(torch.float32) for x, y in runs])*self.scale
            table.index_copy_(0, pos.index_select(0, sidx), got)  # got aligns with srows
        self.calls+=1; self.rows_read+=uniq.numel(); self.parts_touched.update(torch.unique(parts).tolist())
        return table[inv].reshape(*shape, self.rd).flatten(-2)  # [..,16,160]->[..,2560]

def permute_addresses(addrs, seed=777):
    '''Deterministic per-head bijective address permutation (shared by builder + store).'''
    sizes, offs=head_layout()
    S=torch.tensor(sizes); O=torch.tensor(offs)
    A=[]; B=[]
    for h,(s,o) in enumerate(zip(sizes, offs)):
        a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
        b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
        assert a%s!=0, 'A must be coprime to prime head size'
        A.append(a); B.append(b)
    A=torch.tensor(A); B=torch.tensor(B)
    f=addrs.long().cpu()
    H=f.shape[-1]
    O=O[:H]; A=A[:H]; B=B[:H]; S=S[:H]
    return O+((f-O)*A+B)%S

class RandomPLE:
    '''Calibrated per-head deterministic control: same global address -> same 160-d row, every call.
    Means/stds are per-head scalars measured from real rows in the frozen working set.
    16 rows concatenate to 2560-d. No 0.06 fallback.'''
    def __init__(self, head_means, head_stds, seed=0, row_dim=160):
        assert len(head_means)==16 and len(head_stds)==16, 'need 16 per-head stats'
        self.means=[float(m) for m in head_means]
        self.stds=[float(s) for s in head_stds]
        assert all(s>0 for s in self.stds), 'stds must be positive (calibrated)'
        self.seed=seed; self.rd=row_dim
        _sizes,_offs=head_layout()
        self._S=list(_sizes); self._O=list(_offs)
    def _head_of(self, a):
        for h in range(16):
            if self._O[h]<=a<self._O[h]+self._S[h]: return h
        raise ValueError('address outside head ranges')
    def _rows_for(self, uniq, heads):
        rows=[]
        for a,h in zip(uniq.tolist(), heads.tolist()):
            g=torch.Generator(); g.manual_seed((self.seed*1000003+int(a))%2**63)
            rows.append(torch.randn(self.rd, generator=g)*self.stds[h]+self.means[h])
        return torch.stack(rows)
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)
        heads=torch.tensor([self._head_of(int(a)) for a in uniq.tolist()])
        table=self._rows_for(uniq, heads)
        return table[inv].reshape(*shape, self.rd).flatten(-2)

def calibrate_head_stats(compact, n_per_head=4096, seed=0):
    '''Measure per-head mean/std from representative real rows in the compact working set.
    Samples n_per_head rows per head from the frozen prefix (train + full-val).'''
    sizes, offs=head_layout()
    addrs=compact.addrs.to(torch.int64)
    g=torch.Generator().manual_seed(seed)
    means=[]; stds=[]
    for h in range(16):
        lo=offs[h]; hi=offs[h]+sizes[h]
        pos=torch.nonzero((addrs>=lo)&(addrs<hi)).flatten()
        assert len(pos)>=n_per_head, f'head {h} only {len(pos)} rows'
        pick=pos[torch.randint(0,len(pos),(n_per_head,),generator=g)]
        rows=compact.rows.index_select(0, pick).to(torch.float32)*compact.scale
        means.append(float(rows.mean()))
        stds.append(float(rows.std(correction=0)))
    print('calibrated head means:', [round(m,6) for m in means])
    print('calibrated head stds:', [round(s,6) for s in stds])
    return means, stds

class PermutedPLE:
    '''Bijective per-head permutation: off_h + ((a-off_h)*A_h + B_h) % size_h.
    Preserves head ranges (sizes are prime, A_h % size_h != 0 so gcd=1 i.e. coprime) and table distribution.
    Wraps a compact working-set cache, never /kaggle/input random reads during training.'''
    def __init__(self, base, seed=777):
        self.b=base; self.seed=seed
        sizes, offs=head_layout()
        self.S=torch.tensor(sizes); self.O=torch.tensor(offs)
        A=[]; B=[]
        for h,(s,o) in enumerate(zip(sizes, offs)):
            a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
            b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
            assert a%s!=0, 'A must be coprime to prime head size'
            A.append(a); B.append(b)
        self.A=torch.tensor(A); self.B=torch.tensor(B)
    def lookup(self, indices):
        H=indices.shape[-1]; f=indices.long().cpu()
        O=self.O[:H]; A=self.A[:H]; B=self.B[:H]; S=self.S[:H]
        return self.b.lookup((O+((f-O)*A+B)%S).to(indices.device) if indices.is_cuda else (O+((f-O)*A+B)%S))

class CompactPLE:
    '''Compact working set for one frozen token prefix: sorted int32 addresses + fp8 rows (mmap).
    Bit-exact vs MountPLE: same bytes, same scale, same dequant formula. No 48.7 GiB traffic.'''
    def __init__(self, directory):
        import json as _json
        d=Path(directory)
        meta=_json.loads((d/'compact.json').read_text())
        n=meta['address_count']
        self.addrs=torch.from_file(str(d/'addrs.u32'), shared=True, size=n, dtype=torch.int32)
        raw=torch.from_file(str(d/'rows.u8'), shared=True, size=n*meta['row_dim'], dtype=torch.uint8)
        self.rows=raw.view(torch.float8_e4m3fn).view(n, meta['row_dim'])
        self.scale=float(meta['scale']); self.meta=meta
        self.ple_revision=meta.get('ple_revision')
        print('compact PLE: %d rows, %.2f GiB mapped, scale=%g' % (n, (d/'rows.u8').stat().st_size/1024**3, self.scale))
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        assert bool((flat>=0).all()) and int(flat.max())<2**31
        f32=flat.to(torch.int32)
        pos=torch.searchsorted(self.addrs, f32)
        posc=pos.clamp(max=len(self.addrs)-1)
        assert bool((self.addrs[posc]==f32).all()), 'compact miss: address outside frozen prefix'
        out=self.rows.index_select(0, posc).to(torch.float32)*self.scale
        return out.reshape(*shape, self.rows.shape[-1]).flatten(-2)



print('stores ok; manifests:', [str(p) for p in find_ple_manifests()])


In [ ]:
# Cell 7 — tokenizer verification: SEMANTIC id-space check (SPEC #15)
# Pinned-rev forensics: both model.vocab = 248044 entries with 0 id diffs; source-only
# ids are 7 audio added-tokens (248070-248076) above the target max: no collision.
# Config eos = 248044 (<|endoftext|>) on BOTH. AutoTokenizer.eos_token_id may report
# chat <|im_end|> 248046 instead: a chat-template default, NOT the PLE-training
# terminator. Hashing keeps training constants (vocab 248320 / eos 248044 / seed 1234).
import json
from transformers import AutoTokenizer
from huggingface_hub import HfApi, hf_hub_download

tok=secret_value_0
api=HfApi(token=tok)
trev='15852e8c16360a2fea060d615a32b45270f8a8fc'; srev='236dfdf285828023ca3bcd3f37366c58a3469b13'
print('target rev', trev[:12], '| source rev', srev[:12])
tt=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev)
st=AutoTokenizer.from_pretrained(C.SOURCE_ID, token=tok, revision=srev)
def raw_vocab(path):
    tj=json.load(open(path, encoding='utf-8'))
    m=dict(tj['model']['vocab'])
    for a in tj.get('added_tokens', []): m[a['content']]=a['id']
    return m
tm=raw_vocab(hf_hub_download(C.TARGET_ID,'tokenizer.json',revision=trev,token=tok))
sm=raw_vocab(hf_hub_download(C.SOURCE_ID,'tokenizer.json',revision=srev,token=tok))
diff={k for k in tm if k in sm and tm[k]!=sm[k]}
extra_t={k for k in tm if k not in sm}
smax=max(tm.values())
src_only={k: sm[k] for k in sm if k not in tm}
print('mapping diffs:', len(diff), '| target-only:', len(extra_t), '| source-only:', len(src_only))
assert not diff and not extra_t, 'target id space must match source ids exactly (native addressing)'
assert all(v>smax for v in src_only.values()), 'source-only ids must sit above target range'
assert sm.get('<|endoftext|>')==C.EOS and tm.get('<|endoftext|>')==C.EOS, 'training eos must be <|endoftext|> both sides'
probe='The quick brown fox 0123456789 function print(){} <|im_start|>x<|im_end|>'
assert tt.encode(probe, add_special_tokens=False)==st.encode(probe, add_special_tokens=False), 'probe encodings differ — STOP'
print('NATIVE token-ID addressing VALID: identical ids; training terminator eos =', C.EOS)

assert api.model_info(C.TARGET_ID).sha == trev
import hashlib
tp=hf_hub_download(C.TARGET_ID, 'tokenizer.json', revision=trev, token=tok)
sp=hf_hub_download(C.SOURCE_ID, 'tokenizer.json', revision=srev, token=tok)
assert hashlib.sha256(open(tp, 'rb').read()).hexdigest() == '5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42'
assert hashlib.sha256(open(sp, 'rb').read()).hexdigest() == '0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3'
assert json.load(open(tp))['model']['merges'] == json.load(open(sp))['model']['merges']


In [ ]:
# Cell 9 — frozen Qwen3.5-2B via full-config VLM-compat CausalLM (vision frozen, text path): float16 FIRST
import os, torch
from transformers import AutoConfig, AutoModelForCausalLM

tok=secret_value_0
from huggingface_hub import HfApi
trev='15852e8c16360a2fea060d615a32b45270f8a8fc'
cfg=AutoConfig.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)
assert getattr(cfg,'model_type',None)=='qwen3_5', getattr(cfg,'model_type',None)
tconf=cfg.text_config  # dims ONLY — never pass as config= (strips auto_map, breaks class resolution)
print('hidden',tconf.hidden_size,'layers',tconf.num_hidden_layers,'vocab',tconf.vocab_size,'arch',type(cfg).__name__)
assert tconf.hidden_size==2048 and tconf.num_hidden_layers==24
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)

try:
    from transformers.models.qwen3_5 import Qwen3_5ForCausalLM as _Q
    print('direct qwen3_5 import ok:', _Q.__name__)
except Exception:
    import traceback; traceback.print_exc()
    raise RuntimeError('qwen3_5 modeling import failed — true cause above')
ACTIVE_DTYPE=None; model=None
for dt in [torch.float16]:
    try:
        m=AutoModelForCausalLM.from_pretrained(C.TARGET_ID, revision=trev, token=tok,
            trust_remote_code=True, device_map={'':0}, torch_dtype=dt, low_cpu_mem_usage=True)
        m.eval(); m.requires_grad_(False)
        assert sum(1 for p in m.parameters() if p.requires_grad)==0
        ids=tokenizer('The quick brown fox jumps over the lazy dog. '*8, return_tensors='pt').input_ids[:,:64].cuda()
        with torch.inference_mode():
            lg=m(input_ids=ids,use_cache=False).logits
        assert torch.isfinite(lg.float()).all(), 'non-finite logits'
        model=m; ACTIVE_DTYPE=dt; print(f'frozen load OK in {dt} footprint={m.get_memory_footprint()/1024**3:.2f} GiB')
        print('model class:', type(m).__name__, '| decoder blocks:', len(decoder_layers(m)))
        del ids, lg; break
    except Exception as e:
        print(f'{dt} rejected: {str(e)[:200]}')
        try: del m
        except Exception: pass
assert model is not None and ACTIVE_DTYPE is not None
print('backbone = ACTIVE_DTYPE:', ACTIVE_DTYPE)
print('reader params/compute = FP32 (AdamW FP32; norms/scores FP32)')
print('PLE dequant/output = FP32')
print('reader residual output is cast back to backbone dtype')

In [ ]:
# Cell 8 — immutable validation artifacts (built ONCE, never inside training) + FineWeb-Edu-only stream
import hashlib, json
from array import array
from pathlib import Path
from datasets import load_dataset

WORK=Path('/kaggle/working/qwengram-2b'); WORK.mkdir(parents=True, exist_ok=True)
VALDIR=WORK/'val-frozen-v1'; VALDIR.mkdir(exist_ok=True)

def _write_val(name, tokens_u32, meta_extra):
    tp=VALDIR/f'tokens-{name}.uint32le'; mp=VALDIR/f'validation-{name}.json'
    raw=tokens_u32.tobytes(); digest=hashlib.sha256(raw).hexdigest()
    meta={'name':name,'token_count':len(tokens_u32),'seq':512,'count':len(tokens_u32)//512,
            'tokens_sha256':digest, **meta_extra}
    if mp.exists():
        old=json.loads(mp.read_text())
        if old!=meta or tp.read_bytes()!=raw:
            raise RuntimeError(f'Immutable validation {name} differs — refusing to overwrite')
        print(f"reuse frozen val-{name} sha={digest[:16]} n={len(tokens_u32)}"); return meta
    tp.write_bytes(raw); mp.write_text(json.dumps(meta,indent=2)); print(f'wrote frozen val-{name} sha={digest[:16]}')
    return meta

def build_validation_artifacts(tok):
    '''Prefix-consistent: stream 524288 FineWeb-Edu tokens once; fast = first 65536 slice.'''
    from huggingface_hub import HfApi
    import os
    api=HfApi(token=secret_value_0)
    drev=api.dataset_info(C.DATASET_ID).sha; trev2=api.model_info(C.TARGET_ID).sha
    need_full=VALDIR/'validation-full.json'; need_fast=VALDIR/'validation-fast.json'
    if need_full.exists() and need_fast.exists():
        return json.loads(need_fast.read_text()), json.loads(need_full.read_text())
    ds=load_dataset(C.DATASET_ID, C.DATASET_CONFIG, split='train', streaming=True, revision=drev)
    arr=array('I'); docs=0
    for row in ds:
        docs+=1; t=row.get('text') or ''
        if t.strip(): arr.extend(tok.encode(t, add_special_tokens=False)); arr.append(C.EOS)  # training terminator
        if len(arr)>=C.VAL_FULL_TOKENS: del arr[C.VAL_FULL_TOKENS:]; break
    assert len(arr)==C.VAL_FULL_TOKENS, len(arr)
    base={'dataset':C.DATASET_ID,'config':C.DATASET_CONFIG,'dataset_rev':drev,'target_rev':trev2,'docs':docs,'skip_docs':docs}
    fast_arr=array('I', arr[:C.VAL_FAST_TOKENS])
    mf=_write_val('fast', fast_arr, base); mF=_write_val('full', arr, base)
    return mf, mF

def load_validation(name):
    m=json.loads((VALDIR/f'validation-{name}.json').read_text())
    raw=(VALDIR/f'tokens-{name}.uint32le').read_bytes()
    assert hashlib.sha256(raw).hexdigest()==m['tokens_sha256'], 'val checksum mismatch'
    a=array('I'); a.frombytes(raw)
    import torch
    return torch.tensor(a,dtype=torch.long).view(-1,512), m

class FineWebEduStream:
    '''FIRST experiment: FineWeb-Edu only (35B parity). No Stack/UltraChat/Cosmopedia yet.'''
    def __init__(self, tok, skip_docs, dataset_rev):
        '''dataset_rev MUST be the pinned rev from frozen validation metadata — never re-resolve latest.'''
        from datasets import load_dataset
        assert dataset_rev, 'pass dataset_rev from validation-full.json'
        self.ds=load_dataset(C.DATASET_ID, C.DATASET_CONFIG, split='train', streaming=True, revision=dataset_rev).skip(skip_docs)
        self.it=iter(self.ds)  # streaming datasets are not iterators themselves
        self.tok=tok
        self.documents_read=0
    def __iter__(self): return self
    def __next__(self):
        while True:
            t=(next(self.it) or {}).get('text') or ''
            self.documents_read+=1
            if str(t).strip(): return str(t)

def freeze_stream_tokens(tokenizer, stream, max_tokens, seq=512):
    '''EXACT token prefix train_reader consumes: same encode, same C.EOS, same chunking.
    Shared by the trainer and the compact-cache builder so both see identical tokens.'''
    buf = []
    out = []
    seen = 0
    while seen < max_tokens:
        while len(buf) < seq:
            buf += tokenizer.encode(next(stream), add_special_tokens=False) + [C.EOS]
        out.append(buf[:seq])
        buf = buf[seq:]
        seen += seq
    return torch.tensor(out, dtype=torch.long)

print('build with build_validation_artifacts(tok); read with load_validation("fast"/"full")')

In [ ]:
# Reuse and verify the exact frozen validation bytes.
import hashlib, shutil
_hits = list(Path('/kaggle/input').rglob('transfer-shas.json'))
assert len(_hits) == 1
_man = json.loads(_hits[0].read_text())
for _name in ('frozen-v1__tokens-fast.uint32le', 'frozen-v1__tokens-full.uint32le',
              'frozen-v1__validation-fast.json', 'frozen-v1__validation-full.json'):
    _src = _hits[0].parent / _name
    assert hashlib.sha256(_src.read_bytes()).hexdigest() == _man[_name]['sha256']
    _dst = VALDIR / _name.split('__', 1)[1]
    shutil.copyfile(_src, _dst)
    assert hashlib.sha256(_dst.read_bytes()).hexdigest() == _man[_name]['sha256']
# Cell 11b — validation prep: build immutable artifacts ONCE, then load both (idempotent)
mf, mfull = build_validation_artifacts(tokenizer)
val_fast, _ = load_validation("fast")
val_full, _ = load_validation("full")

print("validation ready")
print("fast:", mf["tokens_sha256"])
print("full:", mfull["tokens_sha256"])

from pathlib import Path
import sys
STAGE = _hits[0].parent
for name, record in _man.items():
    if name.startswith(('frozen-v1__', 'calib__')) and name.endswith(('.json', '.uint32le')):
        path = STAGE / name
        assert path.stat().st_size == record['size']
        assert hashlib.sha256(path.read_bytes()).hexdigest() == record['sha256']
for name in ('code__config.py', 'code__frozen_data.py'):
    path = STAGE / name
    assert hashlib.sha256(path.read_bytes()).hexdigest() == _man[name]['sha256']
    dest = WORK / name.split('__', 1)[1]
    shutil.copyfile(path, dest)
sys.path.insert(0, str(WORK))
import frozen_data
hellaswag, hs_meta = frozen_data.hellaswag_rows(1000, secret_value_0)
lambada, lam_meta = frozen_data.lambada_rows(1000, secret_value_0)
assert len(hellaswag) == len(lambada) == 1000
EVAL_MANIFEST = {'full': mfull['tokens_sha256'], 'fast': mf['tokens_sha256'],
                 'domains': {d: _man['frozen-v1__tokens-' + d + '.uint32le']['sha256']
                             for d in ('general', 'code', 'math', 'scientific', 'multilingual')},
                 'calibration': _man['calib__calib-750k-blocks.uint32le']['sha256'],
                 'hellaswag': hashlib.sha256(json.dumps(hellaswag, sort_keys=True).encode()).hexdigest(),
                 'lambada': hashlib.sha256(json.dumps(lambada, sort_keys=True).encode()).hexdigest(),
                 'hellaswag_source': hs_meta, 'lambada_source': lam_meta}
(WORK / 'evaluation-streams.json').write_text(json.dumps(EVAL_MANIFEST, indent=2))
print('frozen evaluation streams', json.dumps(EVAL_MANIFEST), flush=True)

for name in ('code__evaluate.py', 'code__bootstrap.py'):
    src = STAGE / name
    assert hashlib.sha256(src.read_bytes()).hexdigest() == _man[name]['sha256']
    shutil.copyfile(src, WORK / name.split('__', 1)[1])
eval_hits = list(Path('/kaggle/input').rglob('evaluation-streams.json'))
assert len(eval_hits) == 1
prior_eval = json.loads(eval_hits[0].read_text())
for key in ('full', 'fast', 'domains', 'calibration', 'hellaswag', 'lambada'):
    assert EVAL_MANIFEST[key] == prior_eval[key], key


In [ ]:
import hashlib, json, shutil, time
from pathlib import Path

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

assert len(find_ple_manifests()) == 11
master = MountPLE()
cache_dir = WORK / 'arb-cache'
cache_dir.mkdir(parents=True, exist_ok=True)
for name, srcname in (('compact.json', 'compact__compact.json'),
                      ('addrs.u32', 'compact__addrs.u32'),
                      ('rows.u8', 'compact__rows.u8')):
    src = STAGE / srcname
    assert src.stat().st_size == _man[srcname]['size']
    assert sha(src) == _man[srcname]['sha256']
    dst = cache_dir / name
    if not dst.exists() or sha(dst) != _man[srcname]['sha256']:
        shutil.copyfile(src, dst)
    assert sha(dst) == _man[srcname]['sha256']
store = CompactPLE(cache_dir)
assert store.ple_revision == '236dfdf285828023ca3bcd3f37366c58a3469b13'
g = torch.Generator().manual_seed(0)
sample = store.addrs[torch.randint(0, len(store.addrs), (2048,), generator=g)].reshape(128, 16)
maxdiff = (store.lookup(sample) - master.lookup(sample)).abs().max().item()
assert maxdiff == 0.0
cache_meta = store.meta
print('arb compact cache verified', cache_meta['rows_sha256'],
      cache_meta['addrs_sha256'], maxdiff, flush=True)


In [ ]:
from safetensors.torch import load_file
summary_hits = list(Path('/kaggle/input').rglob('reader-15m-summary.json'))
assert len(summary_hits) == 1
reader_summary = json.loads(summary_hits[0].read_text())
reader_path = summary_hits[0].parent / 'reader-15000064.safetensors'
assert sha(reader_path) == reader_summary['reader_sha256']
resume_path = summary_hits[0].parent / 'reader-real-15m-15000064.pt'
assert sha(resume_path) == reader_summary['checkpoint_sha256']
resume = torch.load(resume_path, map_location='cpu', weights_only=False)
assert resume['seen'] == 15000064 and resume['cfg'] == ((2, 8), 1)
assert resume['provenance']['train_stream_sha256'] == json.loads((summary_hits[0].parent / 'reader-train-15m.json').read_text())['sha256']
inj = ReaderInjection(model, [2, 8], C.MEM_DIM, C.HIDDEN, 1, C.GAMMA_INIT).to('cuda')
weights = load_file(str(reader_path), device='cpu')
assert set(weights) == set(inj.state_dict())
assert all(torch.equal(weights[k], resume['reader'][k]) for k in weights)
inj.load_state_dict({k: v.to('cuda') for k, v in weights.items()})
inj.requires_grad_(False)
inj.set_memory(None)
assert not any(p.requires_grad for p in model.parameters())
GAMMAS = {site: float(inj.readers[site].gamma.detach()) for site in ('2', '8')}
UPDATE_NORMS = {}
for key, value in resume['reader'].items():
    if key.endswith(('keys.0.weight', 'value.weight', 'beta')):
        UPDATE_NORMS[key] = float((value.float() - resume['init_state'][key].float()).norm())
print('15M frozen reader loaded', GAMMAS, flush=True)


In [ ]:
# Frozen arbitration math, verbatim from qwen35-08b/tools/arb_stage_src.py.
# Only w, b, alpha2_raw are parameters (~2K). Gate feature uses h8.detach().
# IDX2: alpha2 in [0.75, 1.75], init 1.25. IDX8: alpha8_t = 0.5*sigmoid(w.RMSNorm(h8)+b).
import math
import torch
from torch import nn
ALPHA2_LO = 0.75
ALPHA2_SPAN = 1.0
B_INIT = -1.0986122886681098
def rms_norm(x, eps=1e-6):
    return x.float().mul(torch.rsqrt(x.float().square().mean(-1, keepdim=True) + eps)).to(x.dtype)
class MemoryArbitration(nn.Module):
    def __init__(self, hidden=2048):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(hidden))
        self.b = nn.Parameter(torch.tensor(float(B_INIT)))
        self.alpha2_raw = nn.Parameter(torch.tensor(0.0))
    def alpha2(self):
        return ALPHA2_LO + ALPHA2_SPAN * torch.sigmoid(self.alpha2_raw)
    def alpha8(self, h8):
        hn = rms_norm(h8.detach().float())
        logit = (hn * self.w.float()).sum(-1) + self.b.float()
        return 0.5 * torch.sigmoid(logit)
    def param_count(self):
        return sum(p.numel() for p in self.parameters())
class ArbHooks(nn.Module):
    # Frozen readers give aug = h + gamma*o, so delta = alpha*(aug-h) applies the
    # effective scale without touching gamma. Only arb params receive gradients.
    def __init__(self, model, frozen, arb, decoder_fn):
        super().__init__()
        self.arb = arb
        self.fr2 = frozen.readers["2"]
        self.fr8 = frozen.readers["8"]
        self.fixed_alpha2 = None
        self.memory = None
        self.cur_alpha8 = None
        self.last_alpha8 = None
        dec = decoder_fn(model)
        self.handles = [
            dec[2].register_forward_pre_hook(self._hook2(), with_kwargs=True),
            dec[8].register_forward_pre_hook(self._hook8(), with_kwargs=True),
        ]
    def _a2(self):
        if self.fixed_alpha2 is not None:
            return torch.tensor(float(self.fixed_alpha2), device=self.arb.alpha2_raw.device)
        return self.arb.alpha2()
    def _hook2(self):
        def fn(mod, args, kw):
            assert self.memory is not None
            h = args[0] if args else kw["hidden_states"]
            m = self.memory
            if m.shape[1] != h.shape[1]:
                m = m[:, :h.shape[1]]
            aug = self.fr2(h, m)
            a2 = self._a2().to(h.dtype)
            out = h + a2 * (aug - h)
            if args:
                return (out, *args[1:]), kw
            kw["hidden_states"] = out
            return args, kw
        return fn
    def _hook8(self):
        def fn(mod, args, kw):
            assert self.memory is not None
            h = args[0] if args else kw["hidden_states"]
            m = self.memory
            if m.shape[1] != h.shape[1]:
                m = m[:, :h.shape[1]]
            a8 = self.arb.alpha8(h)
            self.cur_alpha8 = a8
            self.last_alpha8 = a8.detach()
            aug = self.fr8(h, m)
            w8 = a8.to(h.dtype).unsqueeze(-1)
            out = h + w8 * (aug - h)
            if args:
                return (out, *args[1:]), kw
            kw["hidden_states"] = out
            return args, kw
        return fn
    def set_memory(self, m):
        self.memory = m
    def close(self):
        [h.remove() for h in self.handles]
        self.handles.clear()


In [ ]:
import math, os, random, torch.nn.functional as F
from array import array

cal_path = STAGE / 'calib__calib-750k-blocks.uint32le'
assert sha(cal_path) == EVAL_MANIFEST['calibration']
a = array('I'); a.frombytes(cal_path.read_bytes())
CAL = torch.tensor(a, dtype=torch.long).view(-1, 512)
assert CAL.shape == (1464, 512)
torch.manual_seed(1234); random.seed(1234)
arb = MemoryArbitration(hidden=2048).to('cuda')
assert abs(float(arb.alpha2().detach()) - 1.25) < 1e-7
assert abs(float(arb.alpha8(torch.zeros(1, 1, 2048, device='cuda')).detach()) - 0.125) < 1e-7
assert arb.w.numel() == 2048 and torch.count_nonzero(arb.w) == 0
params = list(arb.parameters())
assert len(params) == 3 and sum(p.numel() for p in params) == 2050
hooks = ArbHooks(model, inj, arb, decoder_layers)
assert hooks.fixed_alpha2 is None
work = WORK / 'arbitration'
work.mkdir(exist_ok=True)
reader_sha = sha(reader_path)
ple_stats = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in master.part_paths.values()}
model_versions = {n: p._version for n, p in model.named_parameters()}
reader_versions = {n: p._version for n, p in inj.named_parameters()}
torch.cuda.reset_peak_memory_stats(0)
start_time = time.perf_counter()

def lr_at(step, total):
    if step < 32: return 1e-3 * (step + 1) / 32
    x = min((step - 32) / max(1, total - 32), 1.0)
    return 5e-4 * (1 + math.cos(math.pi * x))

def save_arb(tokens, step, opt, schedule):
    path = work / ('linear-%d.pt' % tokens)
    tmp = path.with_suffix('.pt.tmp')
    state = {'alpha2_raw': arb.alpha2_raw.detach().cpu(),
             'w': arb.w.detach().cpu(), 'b': arb.b.detach().cpu(),
             'optimizer': opt.state_dict(), 'scheduler': schedule,
             'tokens': tokens, 'optimizer_step': step,
             'rng_torch': torch.get_rng_state(), 'rng_cuda': torch.cuda.get_rng_state_all(),
             'rng_python': random.getstate(),
             'reader_sha256': reader_sha,
             'calibration_sha256': EVAL_MANIFEST['calibration'],
             'compact_cache_sha256': cache_meta['rows_sha256'],
             'address_mapping_sha256': cache_meta['addrs_sha256'],
             'target_revision': '15852e8c16360a2fea060d615a32b45270f8a8fc', 'tokenizer_sha256': '5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42',
             'ple_revision': '236dfdf285828023ca3bcd3f37366c58a3469b13', 'injection_idx': [2, 8],
             'loss': 'causal CE + 2*(mean(alpha_late)-0.125)^2'}
    with tmp.open('wb') as f:
        torch.save(state, f); f.flush(); os.fsync(f.fileno())
    test = torch.load(tmp, map_location='cpu', weights_only=False)
    assert test['tokens'] == tokens and test['reader_sha256'] == reader_sha
    assert test['optimizer_step'] == step and len(test['optimizer']['state']) == 3
    assert all(torch.equal(test[k], state[k]) for k in ('alpha2_raw', 'w', 'b'))
    os.replace(tmp, path)
    print('ARB_CHECKPOINT', tokens, sha(path), flush=True)
    return path

for first, last in ((0, 978), (978, 1464)):
    opt = torch.optim.AdamW(params, lr=1e-3, weight_decay=0.0)
    assert {id(p) for group in opt.param_groups for p in group['params']} == {id(p) for p in params}
    for step in range(first, last):
        lr = lr_at(step - first, last - first)
        for group in opt.param_groups: group['lr'] = lr
        ids = CAL[step].unsqueeze(0)
        hooks.set_memory(store.lookup(addresses(ids)).to('cuda'))
        opt.zero_grad(set_to_none=True)
        logits = model(input_ids=ids.to('cuda'), use_cache=False).logits
        ce = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.shape[-1]),
                             ids.to('cuda')[:, 1:].reshape(-1))
        a8 = hooks.cur_alpha8
        loss = ce + 2.0 * (a8.float().mean() - 0.125).square()
        assert torch.isfinite(loss)
        loss.backward()
        assert not any(p.grad is not None for p in model.parameters())
        assert not any(p.grad is not None for p in inj.parameters())
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
        if (step + 1) % 100 == 0:
            print('arb step', step + 1, 'CE', float(ce.detach()),
                  'early', float(arb.alpha2().detach()),
                  'late', float(a8.detach().mean()), flush=True)
    save_arb(last * 512, last, opt, {'leg_start': first, 'leg_end': last,
             'warm_steps': 32, 'lr_peak': 1e-3, 'last_lr': lr})
    assert ple_stats == {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in master.part_paths.values()}
    assert model_versions == {n: p._version for n, p in model.named_parameters()}
    assert reader_versions == {n: p._version for n, p in inj.named_parameters()}
arb_time = time.perf_counter() - start_time
arb_peak = torch.cuda.max_memory_allocated(0) / 1024**3
arb.requires_grad_(False)
hooks.close(); inj.set_memory(None)
assert abs(float(arb.alpha2().detach()) - 1.25) > 1e-6
print('LINEAR750_COMPLETE', arb_time, arb_peak, flush=True)


In [ ]:
import evaluate, bootstrap
from random import Random

val = val_full
domains = {d: frozen_data.load_blocks(STAGE / ('frozen-v1__tokens-%s.uint32le' % d), EVAL_MANIFEST['domains'][d])
           for d in ('general', 'code', 'math', 'scientific', 'multilingual')}

def score_static(label, zero_gamma):
    old = {k: p.detach().clone() for k, p in inj.named_parameters() if k.endswith('gamma')}
    if zero_gamma:
        with torch.no_grad():
            for k, p in inj.named_parameters():
                if k in old: p.zero_()
    result = evaluate.eval_all_static(model, inj, store, addresses, tokenizer,
                                      val, domains, hellaswag, lambada,
                                      time.perf_counter(), 1e12)
    with torch.no_grad():
        for k, p in inj.named_parameters():
            if k in old: p.copy_(old[k])
    assert len(result['val_blocks']) == 1024
    assert len(result['hs_correct']) == len(result['lam_correct']) == 1000
    (work / ('eval-%s.json' % label)).write_text(json.dumps(result))
    print('EVAL', label, result['val_nll'], result['domain_mean'], flush=True)
    return result

stock = score_static('stock', True)
raw = score_static('raw-15m', False)
inj.set_memory(None)
hooks = ArbHooks(model, inj, arb, decoder_layers)
final = evaluate.eval_all(model, hooks, store, addresses, tokenizer,
                          val, domains, hellaswag, lambada,
                          time.perf_counter(), 1e12)
hooks.close()
assert len(final['val_blocks']) == 1024
assert len(final['hs_correct']) == len(final['lam_correct']) == 1000
assert all(len(final['dom_blocks'][d]) == 64 for d in domains)
(work / 'eval-linear750.json').write_text(json.dumps(final))

def paired(a, b, name):
    out = bootstrap.contrast(name, a, b)
    rng = Random(1234)
    dd = {d: [x['sum']/x['n'] - y['sum']/y['n']
              for x, y in zip(a['dom_blocks'][d], b['dom_blocks'][d])]
          for d in sorted(domains)}
    reps = sorted(sum(sum(v[rng.randrange(len(v))] for _ in v)/len(v)
                      for v in dd.values())/len(dd) for _ in range(10000))
    out['domain_mean_nll'] = {'mean': sum(sum(v)/len(v) for v in dd.values())/len(dd),
                              'lo': reps[250], 'hi': reps[9749]}
    out['paired_deltas'] = {'val_nll': [x['sum']/x['n'] - y['sum']/y['n']
                                       for x, y in zip(a['val_blocks'], b['val_blocks'])],
                            'domains': dd,
                            'hellaswag_acc': [x-y for x,y in zip(a['hs_correct'], b['hs_correct'])],
                            'lambada_acc': [x-y for x,y in zip(a['lam_correct'], b['lam_correct'])],
                            'lambada_nll': [x-y for x,y in zip(a['lam_nlls'], b['lam_nlls'])]}
    return out

contrasts = {'raw_minus_stock': paired(raw, stock, 'raw - stock'),
             'linear_minus_stock': paired(final, stock, 'linear750 - stock'),
             'linear_minus_raw': paired(final, raw, 'linear750 - raw')}
(work / 'bootstrap.json').write_text(json.dumps(contrasts))

def update_norms():
    layers = decoder_layers(model)
    captured = {}
    handles = []
    for idx in (2, 8):
        def before(mod, args, kwargs, site=idx):
            captured[site] = (args[0] if args else kwargs['hidden_states']).detach().clone()
        handles.append(layers[idx].register_forward_pre_hook(before, with_kwargs=True, prepend=True))
    active = ArbHooks(model, inj, arb, decoder_layers)
    sums = {site: {'reader': 0.0, 'effective': 0.0, 'relative': 0.0} for site in (2, 8)}
    count = 0
    with torch.inference_mode():
        for block in val[:64]:
            ids = block.unsqueeze(0)
            memory = store.lookup(addresses(ids)).to('cuda')
            active.set_memory(memory)
            model(input_ids=ids.to('cuda'), use_cache=False)
            for site in (2, 8):
                h = captured[site]
                delta = inj.readers[str(site)](h, memory).float() - h.float()
                alpha = arb.alpha2() if site == 2 else arb.alpha8(h)
                effective = delta * alpha.float().unsqueeze(-1) if site == 8 else delta * alpha.float()
                sums[site]['reader'] += float(delta.norm(dim=-1).mean())
                sums[site]['effective'] += float(effective.norm(dim=-1).mean())
                sums[site]['relative'] += float((effective.norm(dim=-1) / h.float().norm(dim=-1).clamp_min(1e-6)).mean())
            count += 1
    active.close()
    for handle in handles: handle.remove()
    return {str(site): {key: value/count for key, value in vals.items()}
            for site, vals in sums.items()}

RESIDUAL_NORMS = update_norms()
summary = {'stock': {k: stock[k] for k in ('val_nll', 'domain_mean', 'domains', 'hs_acc', 'lambada_acc', 'lambada_nll')},
           'raw': {k: raw[k] for k in ('val_nll', 'domain_mean', 'domains', 'hs_acc', 'lambada_acc', 'lambada_nll')},
           'linear750': {k: final[k] for k in ('val_nll', 'domain_mean', 'domains', 'hs_acc', 'lambada_acc', 'lambada_nll', 'alpha8')},
           'early_alpha': float(arb.alpha2().detach()),
           'gammas': GAMMAS,
           'reader_parameter_update_norms': UPDATE_NORMS,
           'reader_residual_update_norms': RESIDUAL_NORMS,
           'effective_early': GAMMAS['2'] * float(arb.alpha2().detach()),
           'effective_late_mean': GAMMAS['8'] * final['alpha8']['full-val']['mean'],
           'arb_time_s': arb_time, 'arb_peak_gib': arb_peak,
           'arb_training_tok_s': 749568 / arb_time,
           'host_rss_mib': rss_mb(),
           'arb_checkpoint_sha256': sha(work / 'linear-749568.pt'),
           'reader_sha256': reader_sha,
           'cache_sha256': cache_meta['rows_sha256'],
           'mapping_sha256': cache_meta['addrs_sha256'],
           'eval_streams_sha256': sha(WORK / 'evaluation-streams.json')}
(work / 'summary.json').write_text(json.dumps(summary, indent=2))
print('QWENGRAM_2B_EVAL_COMPLETE', json.dumps(summary), flush=True)
